<a href="https://colab.research.google.com/github/smagadi/AIML/blob/master/GenAI/Q%26A_PRESALES_TrainedModel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install faiss-cpu
import pandas as pd
import faiss
import numpy as np
import torch
!pip install transformers datasets accelerate
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, T5EncoderModel,DataCollatorForSeq2Seq


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.5/27.5 MB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 29.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 17.8 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2024.10.0
    Uninstalling fsspec-2024.10.0:
      Successfully uninstalled fsspec-2024.10.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2024.10.0 requires fsspec==2024.10.0, but you have fsspec 2024.9.0 which is incompatible.


In [ ]:
# 1. Load Tokenizer and Models
tokenizer = AutoTokenizer.from_pretrained("t5-base")
model = AutoModelForSeq2SeqLM.from_pretrained("t5-base")
encoder_model = T5EncoderModel.from_pretrained("t5-base")

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/892M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [ ]:
df = pd.read_csv("./ps.csv")
df

,QUESTION,ANSWER
0,Service provider should be able to deliver SCA...,We provide SCA service to support 2FA for cred...
1,Sending an OTP to the cardholder on his / her ...,"We support OTP generation, validation and deli..."
2,NaN,OTP will be valid for pre-configured period. O...
3,"If the OTP has expired and has not been used, ...",Resend OTP option will be available to cardhol...
4,In case if OTP is not generated within the sti...,Resend OTP option will be available to cardhol...
5,Validation of an OTP and sending the status of...,"Yes, SCA sends the OTP validation/3DS authenti..."
6,SCA Screen should display the message to the c...,SCA shows the masked mobile and/or email ID on...
7,NaN,SCA UI screen can be customized as per Bank re...
8,OTP delivery in other channel- Email also to b...,OTP delivery to cardholder email ID is support...
9,Email & SMS OTP should be different. Ability t...,Email and SMS OTP can be different. We can tra...


In [ ]:
answer_column_name = "ANSWER"  # Replace with the actual column name

dataset = []
previous_answer = ""

for index, row in df.iterrows():
    question = row["QUESTION"]
    answer = row[answer_column_name]

    if pd.isna(question):
        if previous_answer:
            answer_str = str(answer) if not pd.isna(answer) else ""
            dataset[-1]["ANSWER"] += " " + answer_str
    else:
        # Improved Data Cleaning: Unicase, Remove Spaces, Handle NaN
        question = str(question).lower().strip() if not pd.isna(question) else ""
        answer = str(answer).lower().strip() if not pd.isna(answer) else ""
        dataset.append({"QUESTION": question, "ANSWER": answer})
        previous_answer = answer

df_cleaned = pd.DataFrame(dataset)
df_cleaned = df_cleaned.dropna()  # Drop rows with NaN values if necessary

In [ ]:
df_cleaned

,QUESTION,ANSWER
0,service provider should be able to deliver sca...,we provide sca service to support 2fa for cred...
1,sending an otp to the cardholder on his / her ...,"we support otp generation, validation and deli..."
2,"if the otp has expired and has not been used, ...",resend otp option will be available to cardhol...
3,in case if otp is not generated within the sti...,resend otp option will be available to cardhol...
4,validation of an otp and sending the status of...,"yes, sca sends the otp validation/3ds authenti..."
5,sca screen should display the message to the c...,sca shows the masked mobile and/or email id on...
6,otp delivery in other channel- email also to b...,otp delivery to cardholder email id is support...
7,email & sms otp should be different. ability t...,email and sms otp can be different. we can tra...
8,facility to customize the suppression of email...,we can suppress email otp for certain period o...
9,support for international otp sms to be offered.,we can provide support for international otp s...


TRAIN THE DATA WITH MODEL


In [ ]:
from datasets import Dataset
def preprocess_function(examples):
    inputs = [ex for ex in examples["QUESTION"]]  # Assuming "question" is the column name for questions
    targets = [ex for ex in examples["ANSWER"]]   # Assuming "answer" is the column name for answers
    model_inputs = tokenizer(inputs, max_length=512, truncation=True, padding="max_length")

    # Setup the tokenizer for targets
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(targets, max_length=128, truncation=True, padding="max_length")

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs
# Convert 'dataset' (list) to a pandas DataFrame
df_dataset = Dataset.from_pandas(df_cleaned)

tokenized_datasets = df_dataset.map(preprocess_function, batched=True)


Map:   0%|          | 0/34 [00:00<?, ? examples/s]

/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:4114: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(


In [ ]:
# 4. Split into train and validation sets
train_testvalid = tokenized_datasets.train_test_split(test_size=0.2) # 80% train, 20% validation


In [ ]:
from datasets import DatasetDict # This line imports DatasetDict
# Further split the validation set into validation and test sets (50% each)
test_valid = train_testvalid['test'].train_test_split(test_size=0.5)
# Merge the splits to have train, validation, and test sets
tokenized_datasets = DatasetDict({
    'train': train_testvalid['train'],
    'test': test_valid['test'],
    'validation': test_valid['train']
})

In [ ]:
!pip install transformers datasets accelerate scikit-learn
!pip install evaluate
import evaluate
#!pip install --upgrade datasets
from transformers import TrainingArguments,Trainer,AdamW # Import TrainingArguments here
from transformers import get_scheduler
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 7.5 MB/s eta 0:00:00


In [ ]:
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions[0].argmax(-1)

    # Remove padding tokens
    labels_no_pad = labels[labels != -100]
    preds_no_pad = preds[labels != -100]

    # Load the metrics from the 'evaluate' library
    metric_accuracy = evaluate.load("accuracy")
    metric_f1 = evaluate.load("f1")
    metric_precision = evaluate.load("precision")
    metric_recall = evaluate.load("recall")

    # Calculate metrics
    acc = metric_accuracy.compute(predictions=preds_no_pad, references=labels_no_pad)["accuracy"]
    f1_results = metric_f1.compute(predictions=preds_no_pad, references=labels_no_pad, average='weighted')
    precision_results = metric_precision.compute(predictions=preds_no_pad, references=labels_no_pad, average='weighted')
    recall_results = metric_recall.compute(predictions=preds_no_pad, references=labels_no_pad, average='weighted')
    print(f"Accuracy: {acc}")
    print(f"F1 Score: {f1_results}")
    print(f"Precision: {precision_results}")
    print(f"Recall: {recall_results}")

    return {
        'accuracy': acc,
        'f1': f1_results["f1"],
        'precision': precision_results["precision"],
        'recall': recall_results["recall"]
    }

In [ ]:
# 5. Define training arguments
training_args = TrainingArguments(
    output_dir="./results",
    run_name="my-awesome-run",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,  # Keep only the best 2 checkpoints
    load_best_model_at_end=True, # Load the best model at the end of training
    metric_for_best_model="eval_loss", # Use evaluation loss to determine the best model
    report_to="none",
    learning_rate=5e-5, # AdamW default learning rate
    weight_decay=0.01,  # AdamW default weight decay
    logging_dir='./logs', # Directory for storing logs
    logging_steps=10, # Log every 10 steps
    # ... other training arguments ...
)

/usr/local/lib/python3.10/dist-packages/transformers/training_args.py:1568: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [ ]:
# Create the optimizer
optimizer = AdamW(model.parameters(), lr=training_args.learning_rate, weight_decay=training_args.weight_decay)

# Create the learning rate scheduler
num_training_steps = len(tokenized_datasets["train"]) // training_args.per_device_train_batch_size * training_args.num_train_epochs
lr_scheduler = get_scheduler(
    "linear",
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=num_training_steps,
)

/usr/local/lib/python3.10/dist-packages/transformers/optimization.py:591: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


In [ ]:
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    optimizers=(optimizer, lr_scheduler)  # Pass the optimizer and scheduler
    # ... other Trainer arguments ...
)

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,No log,8.822562,0.091146,0.072833,0.075328,0.091146
2,No log,8.139113,0.088542,0.071435,0.074545,0.088542
3,8.048100,8.087669,0.088542,0.071435,0.074545,0.088542


Accuracy: 0.09114583333333333
F1 Score: {'f1': 0.07283255294910009}
Precision: {'precision': 0.07532770920868348}
Recall: {'recall': 0.09114583333333333}


/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Accuracy: 0.08854166666666667
F1 Score: {'f1': 0.07143537266305124}
Precision: {'precision': 0.07454536044281729}
Recall: {'recall': 0.08854166666666667}


/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Accuracy: 0.08854166666666667
F1 Score: {'f1': 0.07143537266305124}
Precision: {'precision': 0.07454536044281729}
Recall: {'recall': 0.08854166666666667}


/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].


TrainOutput(global_step=12, training_loss=8.045362790425619, metrics={'train_runtime': 86.3544, 'train_samples_per_second': 0.938, 'train_steps_per_second': 0.139, 'total_flos': 49325589135360.0, 'train_loss': 8.045362790425619, 'epoch': 3.0})

In [ ]:
trainer.save_model("./best_model")

WE WILL ADD RAG LATER

In [ ]:
embeddings = []
for _, row in df_cleaned.iterrows():
    question = row['QUESTION']
    inputs = tokenizer(question, return_tensors='pt', padding=True, truncation=True)
    with torch.no_grad():
        outputs = encoder_model(**inputs)
    embedding = outputs.last_hidden_state.mean(dim=1).detach().cpu().numpy()
    embeddings.append(embedding)

embeddings = np.array(embeddings).astype('float32')
embeddings = embeddings.reshape(embeddings.shape[0], embeddings.shape[-1])


Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


In [ ]:
d = encoder_model.config.hidden_size
index = faiss.IndexFlatL2(d)
index.add(embeddings)

In [ ]:
def get_embedding(text):
    inputs = tokenizer(text, return_tensors='pt', padding=True, truncation=True)
    with torch.no_grad():
        outputs = encoder_model(**inputs)
    embedding = outputs.last_hidden_state.mean(dim=1).detach().cpu().numpy()
    embedding = embedding.reshape(1, -1)
    return embedding

In [ ]:
# Load the tokenizer and model
tokenizer = AutoTokenizer.from_pretrained("t5-base")  # Assuming you used 't5-base' during training
model_Trained = AutoModelForSeq2SeqLM.from_pretrained("./best_model")  # Path to the saved model directory

In [ ]:
def generate_responsenew(prompt, top_k=1, threshold=6):
    query_embedding = get_embedding(prompt)
    #print("Query Embedding:", query_embedding)
    distances, indices = index.search(query_embedding.astype('float32'), k=top_k)
    print("Distances:", distances)  # Print the distances
    print("Indices:", indices) # Print the indices
    similar_questions = []
    for i in range(top_k):
      #print("pricessing ")
      try:
          #if distances[0][i] < threshold:
            similar_questions.append({
                "question": df_cleaned.iloc[indices[0][i]]['QUESTION'],
                "answer": df_cleaned.iloc[indices[0][i]]['ANSWER'],
                "distance": distances[0][i]
            })
      except IndexError as e:
            print(f"IndexError at i={i}: {e}")
            print(f"indices shape: {indices.shape}, indices[0]: {indices[0]}, len(df_cleaned): {len(df_cleaned)}")

    print("Similar Questions:", similar_questions)

    if similar_questions:
        context = "\n".join([f"Q: {q['question']}\nA: {q['answer']}" for q in similar_questions])

        prompt_with_context = f"Context: {context}\n\nQuestion: {prompt}\nAnswer:"
        print(prompt_with_context)
        inputs = tokenizer(prompt_with_context, return_tensors="pt", padding=True, truncation=True, max_length=512)
        outputs = model_Trained.generate(**inputs, max_length=512,
                                         temperature=0.7,           # Experiment with values
                            repetition_penalty=1.2,   # Increase to discourage repetition
                            no_repeat_ngram_size=3,    # Prevent repeating n-grams
                            early_stopping=True)
        print("Token IDs:", outputs)
        #response = tokenizer.decode(outputs[0], skip_special_tokens=True)
        response = tokenizer.decode(outputs[0], skip_special_tokens=True)

        return response
    else:
        response = "I'm sorry, I don't have enough information to answer that question."
        return response

In [ ]:
new_question = "Do you support able to connect with the external apis to feed.?".lower().strip()  # Apply cleaning to the new question
answer =generate_responsenew(new_question)
print(answer)

Distances: [[5.0980473 7.050787  7.1938763]]
Indices: [[17  7 18]]
Similar Questions: [{'question': 'able to connect with the external apis to feed in information additional information for decisioning purposes.', 'answer': 'we have customer update api that can be integrated with bank system to update customer mobile number and email id real time basis.', 'distance': 5.0980473}]
Context: Q: able to connect with the external apis to feed in information additional information for decisioning purposes.
A: we have customer update api that can be integrated with bank system to update customer mobile number and email id real time basis.

Question: do you support able to connect with the external apis to feed.?
Answer:
Token IDs: tensor([[    0, 10998,     1]])
True


In [ ]:
new_question = "is the solution certified by NPCI, Visa, MasterCard and AMEX.?".lower().strip()  # Apply cleaning to the new question
response =generate_responsenew(new_question)
print(response)

Distances: [[1.7957664 3.6216617 7.8196726]]
Indices: [[13 12 23]]
Similar Questions: [{'question': 'solution provider should be certified by npci, visa, mastercard and amex.', 'answer': 'certificates attached in the executive summary', 'distance': 1.7957664}, {'question': 'application should comply with all network regulations visa, mastercard, jcb, rupay, amex, npci.', 'answer': 'we support all network regulations visa, mastercard, rupay, jcb, amex and npci.', 'distance': 3.6216617}]
Context: Q: solution provider should be certified by npci, visa, mastercard and amex.
A: certificates attached in the executive summary
Q: application should comply with all network regulations visa, mastercard, jcb, rupay, amex, npci.
A: we support all network regulations visa, mastercard, rupay, jcb, amex and npci.

Question: is the solution certified by npci, visa, mastercard and amex.?
Answer:
Token IDs: tensor([[    0, 10998,     1]])
True


In [ ]:
new_question = "Transaction logs storage.?".lower().strip()  # Apply cleaning to the new question
response =generate_responsenew(new_question)
print(response)

Distances: [[10.368082 10.813004 11.436267]]
Indices: [[20 24  0]]
Similar Questions: [{'question': 'should have the facility to save and share the sms and transaction logs for 5 years through an automated sftp process.', 'answer': 'we can save and share the sms and transactions logs for 5 years through an automated sftp process.', 'distance': 10.368082}, {'question': 'necessary real time alerting and monitoring mechanism for service outages will be provided to sbi card through web portals and tools.', 'answer': 'we can configure real time alerts for service outages as well as 30 min/60 min stats on bank email ids. Bank can also check the transaction SR by login to the SCA(IVS) console.', 'distance': 10.813004}, {'question': 'service provider should be able to deliver sca service to support the 2fa for credit card transactions decisioning.', 'answer': 'we provide sca service to support 2fa for credit and debit card transactions decisioning. currently we are providing sca service to ban

In [ ]:
new_question = "does platform provides MIS reports.?".lower().strip()  # Apply cleaning to the new question
response =generate_responsenew(new_question)
print(response)

Distances: [[8.898542 9.343061 9.36554 ]]
Indices: [[25  0 24]]
Similar Questions: [{'question': 'platform mis and reports to be provided at the time of go-live.', 'answer': 'sca (ivs) and frm (frm) console provide real time dashboard and txn reports. these reports can be viewed and downloaded from the console. bank can access the console through user credentials.', 'distance': 8.898542}, {'question': 'service provider should be able to deliver sca service to support the 2fa for credit card transactions decisioning.', 'answer': 'we provide sca service to support 2fa for credit and debit card transactions decisioning. currently we are providing sca service to bank cards(visa, mastercard, amex and rupay cards)', 'distance': 9.343061}, {'question': 'necessary real time alerting and monitoring mechanism for service outages will be provided to sbi card through web portals and tools.', 'answer': 'we can configure real time alerts for service outages as well as 30 min/60 min stats on bank ema

In [ ]:
new_question = "does platform has a fraud engine.?".lower().strip()  # Apply cleaning to the new question
response =generate_responsenew(new_question)
print(response)

Distances: [[7.6710253 8.1084385 8.152079 ]]
Indices: [[30 32  0]]
Similar Questions: [{'question': 'should have in built fraud engine which can be used for various fraud use cases.', 'answer': 'frm frm risk based authentication is a part of the solution to assess risk involved in the ecom transactions', 'distance': 7.6710253}, {'question': 'service provider should also be able to provide multi-factor authentication using device biometrics etc.', 'answer': 'we have frmity product for out of band multifactor authentication solution where cardholder can be authenticated using multiple authentication options like a. Push notification based authentication b. Push notification + Biometric authentication(Device native authentication) c. Offline and instant OTP authentication. Customer will have to enroll for the service where the device and SIM binding is carried out and is compliant as per the local regulatory requirements ', 'distance': 8.1084385}, {'question': 'service provider should be 

In [ ]:
new_question = "does platform provides has a fraud engine that can be driven by rules.?".lower().strip()  # Apply cleaning to the new question
response =generate_responsenew(new_question)
print(response)

Distances: [[5.737543  6.0748816 6.3362226]]
Indices: [[30  0 32]]
Similar Questions: [{'question': 'should have in built fraud engine which can be used for various fraud use cases.', 'answer': 'frm frm risk based authentication is a part of the solution to assess risk involved in the ecom transactions', 'distance': 5.737543}, {'question': 'service provider should be able to deliver sca service to support the 2fa for credit card transactions decisioning.', 'answer': 'we provide sca service to support 2fa for credit and debit card transactions decisioning. currently we are providing sca service to bank cards(visa, mastercard, amex and rupay cards)', 'distance': 6.0748816}, {'question': 'service provider should also be able to provide multi-factor authentication using device biometrics etc.', 'answer': 'we have frmity product for out of band multifactor authentication solution where cardholder can be authenticated using multiple authentication options like a. Push notification based auth

In [ ]:
new_question = "does platform support multi factor authentication ".lower().strip()  # Apply cleaning to the new question
response =generate_responsenew(new_question)
response

Distances: [[7.924759 8.981295 9.932648]]
Indices: [[32 10  0]]
Similar Questions: [{'question': 'service provider should also be able to provide multi-factor authentication using device biometrics etc.', 'answer': 'we have frmity product for out of band multifactor authentication solution where cardholder can be authenticated using multiple authentication options like a. Push notification based authentication b. Push notification + Biometric authentication(Device native authentication) c. Offline and instant OTP authentication. Customer will have to enroll for the service where the device and SIM binding is carried out and is compliant as per the local regulatory requirements ', 'distance': 7.924759}, {'question': 'the application shall have support for multiple languages processing.', 'answer': 'we support multiple language processing on 3ds screens.', 'distance': 8.981295}, {'question': 'service provider should be able to deliver sca service to support the 2fa for credit card transa

'True'

In [ ]:
new_question = "Are you certified with master card ?".lower().strip()  # Apply cleaning to the new question
response =generate_responsenew(new_question)
response

Distances: [[9.72665]]
Indices: [[13]]
Similar Questions: [{'question': 'solution provider should be certified by npci, visa, mastercard and amex.', 'answer': 'certificates attached in the executive summary', 'distance': 9.72665}]
Context: Q: solution provider should be certified by npci, visa, mastercard and amex.
A: certificates attached in the executive summary

Question: are you certified with master card ?
Answer:
Token IDs: tensor([[    0, 10998,     1]])


'True'